# ClaimWise Insurance Prediction — 12. Logistic Regression (Classification Extension)

## 1. Objective

Add the academic **classification** requirement on top of the existing regression system, without
touching it:

| System | Target | Models | Metrics |
|---|---|---|---|
| Production ClaimWise (notebooks 04-10) | `loss` (continuous) | 5 regressors | MAE, MSE, RMSE, R² |
| Academic extension (this notebook) | `loss_high` (derived binary) | Logistic Regression | Confusion matrix, precision, recall, F1 |

| Item | Location |
|---|---|
| Dataset | `Dataset/01_raw_claimwise_50000.csv` (the real ClaimWise data, unchanged) |
| Features | `Dataset/06_split_X_train.csv`, `06_split_X_test.csv` (existing preprocessing) |
| Shared helpers | `Src/academic_common.py` |

Nothing here replaces or modifies the production loss-prediction model.

## 2. Import Libraries

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Walk upwards from the notebook folder until the ClaimWise project is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "Dataset").is_dir() and (candidate / "Src").is_dir():
            return candidate
    raise FileNotFoundError(f"ClaimWise project root not found above: {start}")


PROJECT_ROOT = find_project_root(Path.cwd())
DATASET_DIR = PROJECT_ROOT / "Dataset"
SRC_DIR = PROJECT_ROOT / "Src"
MODELS_DIR = PROJECT_ROOT / "Models"
OUTPUTS_DIR = PROJECT_ROOT / "Outputs"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root  :", PROJECT_ROOT)
print("Dataset folder:", DATASET_DIR)
print("Src folder    :", SRC_DIR)

from academic_common import (
    CLASSIFICATION_DIR,
    CLASS_TARGET,
    TARGET,
    build_classification_target,
    classification_metrics,
    ensure_dirs,
    plot_confusion_matrix,
    save_figure,
    save_metric_rows,
    style_axes,
)
from model_training_utils import load_prepared_data

ensure_dirs()
print("classification output folder:", CLASSIFICATION_DIR)

Project root  : /Users/padaltiruvinayak/Desktop/Credit ML
Dataset folder: /Users/padaltiruvinayak/Desktop/Credit ML/Dataset
Src folder    : /Users/padaltiruvinayak/Desktop/Credit ML/Src


classification output folder: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/classification


In [3]:
# Src/model_training_utils.py switches matplotlib to the non-interactive Agg backend so the
# command-line scripts can run without a display. Switch back to the inline backend here so
# that every figure below is rendered inside this notebook.
%matplotlib inline

## 3. Classification Target — why it exists and how it is derived

The production ClaimWise target `loss` is **continuous**, so the production system stays a regression
system (MAE / MSE / RMSE / R²) and this notebook does not change it.

The raw dataset was inspected: it contains only `id`, `cat1`–`cat116`, `cont1`–`cont14` and `loss` —
there is **no pre-existing categorical target** (no fraud/claim-status/label column). The academic
classification requirement therefore needs a derived target, built from `loss` itself:

| Item | Value |
|---|---|
| Derived target | `loss_high` (binary) |
| Threshold | **median of the 40,000 training rows only** — calculated and printed below, never hand-picked |
| Positive class `1` | high-loss claim (`loss >= threshold`) |
| Negative class `0` | typical-loss claim (`loss < threshold`) |
| Held-out influence | none — the test rows never take part in choosing the threshold |

The class balance is checked explicitly below, so any imbalance is visible instead of hidden.

In [4]:
raw_columns = list(pd.read_csv(DATASET_DIR / "01_raw_claimwise_50000.csv", nrows=0).columns)
print("Raw columns (", len(raw_columns), "):", raw_columns[:4], "...", raw_columns[-4:])
possible_targets = [
    c for c in raw_columns
    if not (c.startswith("cat") or c.startswith("cont") or c in ("id", "loss"))
]
print("Pre-existing categorical/target columns found:", possible_targets or "NONE")
print("=> The only target in ClaimWise is the continuous column 'loss'.")

Raw columns ( 132 ): ['id', 'cat1', 'cat2', 'cat3'] ... ['cont12', 'cont13', 'cont14', 'loss']
Pre-existing categorical/target columns found: NONE
=> The only target in ClaimWise is the continuous column 'loss'.


In [5]:
y_train_cls, y_test_cls, target_info = build_classification_target()

print("Derived target :", CLASS_TARGET)
print("Threshold       :", round(target_info["threshold"], 4),
      "(median of the", target_info["train_rows"], "training losses)")
print("Positive class  :", target_info["positive_class"])
print("Negative class  :", target_info["negative_class"])
print()
print("Training class counts:", target_info["train_counts"])
print("Test     class counts:", target_info["test_counts"])

train_total = sum(target_info["train_counts"].values())
test_total = sum(target_info["test_counts"].values())
train_share = target_info["train_counts"]["high (1)"] / train_total
test_share = target_info["test_counts"]["high (1)"] / test_total
print(f"Share of high-loss claims: train {train_share:.4f} | test {test_share:.4f}")
print("Balanced?" , "YES - the classes are (near) perfectly balanced by the median split."
      if abs(train_share - 0.5) < 0.01 else "NO - document the imbalance.")

Derived target : loss_high
Threshold       : 2118.86 (median of the 40000 training losses)
Positive class  : 1 = high-loss claim (loss >= threshold)
Negative class  : 0 = typical-loss claim (loss < threshold)

Training class counts: {'typical (0)': 20000, 'high (1)': 20000}
Test     class counts: {'typical (0)': 5037, 'high (1)': 4963}
Share of high-loss claims: train 0.5000 | test 0.4963
Balanced? YES - the classes are (near) perfectly balanced by the median split.


In [6]:
distribution = pd.DataFrame(
    [
        {"split": "train", "class": "0 typical loss", "samples": target_info["train_counts"]["typical (0)"]},
        {"split": "train", "class": "1 high loss", "samples": target_info["train_counts"]["high (1)"]},
        {"split": "test", "class": "0 typical loss", "samples": target_info["test_counts"]["typical (0)"]},
        {"split": "test", "class": "1 high loss", "samples": target_info["test_counts"]["high (1)"]},
    ]
)
display(distribution)

fig, ax = plt.subplots(figsize=(7, 4.5))
pivot = distribution.pivot(index="split", columns="class", values="samples")
pivot.plot(kind="bar", ax=ax, color=["#78716C", "#D4512A"], rot=0)
style_axes(ax, "Derived classification target — class distribution", "", "samples")
ax.legend(title="class")
dist_path = save_figure(fig, CLASSIFICATION_DIR / "classification_class_distribution.png")
print("Saved:", dist_path)

,split,class,samples
0,train,0 typical loss,20000
1,train,1 high loss,20000
2,test,0 typical loss,5037
3,test,1 high loss,4963


Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/classification/classification_class_distribution.png


## 4. Load the Existing ClaimWise Data

The same prepared split the production models use — no other dataset, no re-shuffling, so the
classification extension stays aligned with the regression system.

In [7]:
X_train, X_test, y_train, y_test = load_prepared_data()

checks = {
    "X_train rows == train labels": len(X_train) == len(y_train_cls),
    "X_test rows == test labels": len(X_test) == len(y_test_cls),
    "same feature columns in train and test": list(X_train.columns) == list(X_test.columns),
    "130 input features": X_train.shape[1] == 130,
    "all features numeric": all(pd.api.types.is_numeric_dtype(d) for d in X_train.dtypes),
    "no missing feature values": not (X_train.isna().any().any() or X_test.isna().any().any()),
    "labels are binary {0,1}": set(np.unique(y_train_cls)) == {0, 1},
    "target 'loss' is not a feature": "loss" not in X_train.columns,
}
report = pd.DataFrame([{"check": k, "result": "PASS" if v else "FAIL"} for k, v in checks.items()])
display(report)
failed = [k for k, v in checks.items() if not v]
if failed:
    raise AssertionError(f"Classification data checks failed: {failed}")
print("All", len(checks), "checks passed.")

,check,result
0,X_train rows == train labels,PASS
1,X_test rows == test labels,PASS
2,same feature columns in train and test,PASS
3,130 input features,PASS
4,all features numeric,PASS
5,no missing feature values,PASS
6,"labels are binary {0,1}",PASS
7,target 'loss' is not a feature,PASS


All 8 checks passed.


## 5. Preprocessing and Leakage

The features come from `Dataset/06_split_X_train.csv` / `06_split_X_test.csv`: the production
pipeline (notebook 02) scaled **only the continuous columns** with a `StandardScaler` fitted on the
80% training rows, while the categorical columns remain integer codes (values up to 265).
Using those columns leaks nothing from the test rows — the same guarantee the production regression
models rely on.

Logistic Regression is gradient-based, and the unscaled categorical codes make plain `lbfgs` fail to
converge (the first executed run reported a `ConvergenceWarning`, which is why it is addressed here
instead of hidden). The classifier therefore gets its own `StandardScaler` fitted **on the training
rows only** and applied to the test rows (next cell). This layer affects the classifier only — the
production regression system (notebooks 04-10) is untouched, and `max_iter=1000` is kept high with
the achieved `n_iter_` printed below.

### 5.1 Leakage-free scaling for the classifier

`StandardScaler` learns mean and standard deviation **from the 40,000 training rows only** and is
then applied unchanged to the 10,000 test rows — so no test information enters the fit.

In [8]:
from sklearn.preprocessing import StandardScaler

lr_scaler = StandardScaler().fit(X_train)
X_train_lr = lr_scaler.transform(X_train)
X_test_lr = lr_scaler.transform(X_test)

print("scaler fitted on:", X_train.shape)
print("max |column mean| (train):", round(float(np.abs(X_train_lr.mean(axis=0)).max()), 6))
print("max |column mean| (test): ", round(float(np.abs(X_test_lr.mean(axis=0)).max()), 6))
print("X_test transformed with train statistics:", X_test_lr.shape)

scaler fitted on: (40000, 130)
max |column mean| (train): 0.0
max |column mean| (test):  0.03846
X_test transformed with train statistics: (10000, 130)


In [9]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_lr, y_train_cls)
print(model)
print("n_iter_:", model.n_iter_)

LogisticRegression(max_iter=1000, random_state=42)
n_iter_: [59]


In [10]:
y_pred = model.predict(X_test_lr)
y_prob = model.predict_proba(X_test_lr)[:, 1]

metrics = classification_metrics(y_test_cls, y_pred)
print("Logistic Regression — test-set classification metrics")
for name, value in metrics.items():
    print(f"  {name:10s} {value:.6f}")

metrics_path = save_metric_rows(
    CLASSIFICATION_DIR / "logistic_regression_metrics.csv",
    "Logistic Regression",
    metrics,
)
print("Saved:", metrics_path)

Logistic Regression — test-set classification metrics
  accuracy   0.758300
  precision  0.782763
  recall     0.710054
  f1         0.744638
Saved:                  model  accuracy  precision    recall        f1
0  Logistic Regression    0.7583   0.782763  0.710054  0.744638


In [11]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test_cls, y_pred,
    target_names=["typical loss (0)", "high loss (1)"],
    digits=4,
))

                  precision    recall  f1-score   support

typical loss (0)     0.7383    0.8058    0.7706      5037
   high loss (1)     0.7828    0.7101    0.7446      4963

        accuracy                         0.7583     10000
       macro avg     0.7605    0.7579    0.7576     10000
    weighted avg     0.7604    0.7583    0.7577     10000



In [12]:
cm_path = plot_confusion_matrix(
    y_test_cls,
    y_pred,
    "Logistic Regression — confusion matrix (test set)",
    CLASSIFICATION_DIR / "logistic_regression_confusion_matrix.png",
)
from IPython.display import Image, display as display_image
display_image(filename=str(cm_path))

from sklearn.metrics import confusion_matrix
tn, fp, fn, tp = confusion_matrix(y_test_cls, y_pred, labels=[0, 1]).ravel()
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")

TN=4059 FP=978 FN=1439 TP=3524


In [13]:
from sklearn.metrics import RocCurveDisplay, roc_auc_score

auc = roc_auc_score(y_test_cls, y_prob)
fig, ax = plt.subplots(figsize=(6.5, 5.2))
RocCurveDisplay.from_predictions(y_test_cls, y_prob, ax=ax, curve_kwargs={"color": "#D4512A"}, name="Logistic Regression")
ax.plot([0, 1], [0, 1], linestyle="--", color="#78716C", label="random classifier")
style_axes(ax, f"Logistic Regression ROC curve (AUC = {auc:.4f})", "false positive rate", "true positive rate")
ax.legend(loc="lower right")
roc_path = save_figure(fig, CLASSIFICATION_DIR / "logistic_regression_roc_curve.png")
print("ROC AUC:", round(auc, 4), "| saved:", roc_path)

ROC AUC: 0.8385 | saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/classification/logistic_regression_roc_curve.png


In [14]:
coefficients = pd.DataFrame(
    {"feature": X_train.columns, "coefficient": np.abs(model.coef_[0])}
).sort_values("coefficient", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.bar(coefficients["feature"], coefficients["coefficient"], color="#E58A3A")
style_axes(ax, "Logistic Regression — 10 largest |coefficients| (standardised features)",
           "", "|coefficient| (standardised inputs)")
coef_path = save_figure(fig, CLASSIFICATION_DIR / "logistic_regression_top_coefficients.png")
display(coefficients.reset_index(drop=True))
print("Saved:", coef_path)

,feature,coefficient
0,cat80,0.484810
1,cat111,0.403177
2,cat101,0.378728
3,cat12,0.370801
4,cat53,0.324883
5,cat81,0.278280
6,cat6,0.274426
7,cont12,0.267556
8,cat79,0.266083
9,cont1,0.236124


Saved: /Users/padaltiruvinayak/Desktop/Credit ML/Outputs/classification/logistic_regression_top_coefficients.png


## 7. What the Classification Numbers Mean

All three metrics are computed on the **10,000 held-out test rows** and use `1 = high-loss claim` as
the positive class:

- **Confusion matrix** — four counts (TN, FP, FN, TP) of predicted vs actual class; it is built from
  test predictions only, never from training predictions.
- **Precision** = TP / (TP + FP) — of the claims flagged as high-loss, how many really were.
- **Recall** = TP / (TP + FN) — of the truly high-loss claims, how many were found.
- **F1-score** = harmonic mean of precision and recall; a single number when both matter.
- **Accuracy** = (TP + TN) / all rows — reported for completeness, but the class balance makes
  precision/recall/F1 the more informative measures.

Because the problem is **binary**, plain (binary) averaging is the correct choice — macro/weighted
averaging only matters for more than two classes.

## 8. Conclusion

Actual results of this executed run (10,000 held-out test rows, positive class = high-loss claim):

| Model | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Logistic Regression | 0.7583 | 0.7828 | 0.7101 | 0.7446 |

Classifier input: the 130 production features plus a train-fitted `StandardScaler` layer (§5.1).

Derived target threshold: **2118.86** (median of the 40,000 training losses) — classes are
balanced, so no resampling was needed.

Artifacts: `Outputs/classification/logistic_regression_metrics.csv`,
`logistic_regression_confusion_matrix.png`, `logistic_regression_roc_curve.png`,
`logistic_regression_top_coefficients.png`, `classification_class_distribution.png`.

The Decision Tree classifier is compared in `13_ClaimWise_Decision_Tree_Classification.ipynb`.
The production regression system (notebooks 04-10) is unchanged.